# Modelo entrenado - prueba de inferencia

In [1]:
import pandas as pd
import joblib

In [3]:
# Cargar el modelo entrenado
modelo_pred_diabetes = joblib.load("/home/mcarneiro/master-ucm-tfm/modelos/modelo_diabetes.pkl")

In [20]:
# Una función para mejor presentar los resultados en base a datos nuevos (nuevo paciente)
def presentar_prediccion(prob, thr=0.5, zona_inferior=0.40, zona_superior=0.60, mostrar_score=False):
    """
    prob: probabilidad de diabetes (0..1)
    thr : umbral de decisión elegido (alineado a tu métrica: PR-AUC/F1/Recall, etc.)
    zona_inferior / zona_superior: banda de incertidumbre
    mostrar_score: si True, añade el % al final (redondeado)
    """
    if zona_inferior <= prob <= zona_superior:
        msg = "Resultado INCIERTO"
        accion = "Recomendado ampliar información o repetir medición/chequeo."
        riesgo = "Riesgo moderado"
    else:
        etiqueta = "DIABÉTICO" if prob >= thr else "NO DIABÉTICO"
        if prob < 0.30:
            riesgo = "Riesgo bajo"
            accion = "Mantener hábitos saludables y seguimiento rutinario."
        elif prob < 0.60:
            riesgo = "Riesgo moderado"
            accion = "Considerar chequeo preventivo con profesional."
        else:
            riesgo = "Riesgo alto"
            accion = "Recomendado evaluación clínica prioritaria."
        msg = f"{etiqueta} — {riesgo}"

    if mostrar_score:
        msg += f" (score: {round(prob*100)}%)"

    return msg, accion

In [24]:
nuevo_paciente = pd.DataFrame([{
    "HighBP": 0,               # 1 = hipertensión, 0 = no
    "HighChol": 0,             # colesterol alto
    "Smoker": 0,               # fumador
    "Stroke": 0,               # historial de ictus
    "HeartDiseaseorAttack": 0, # enfermedad cardíaca
    "PhysActivity": 1,         # actividad física
    "Fruits": 1,               # consume frutas
    "Veggies": 1,              # consume verduras
    "NoDocbcCost": 0,          # no dejó de ir al médico por coste
    "DiffWalk": 0,             # sin dificultad para caminar
    "Sex": 0,                  # 1=Hombre, 0=Mujer
    "BMI": 27,                 # índice de masa corporal
    "MentHlth": 0,             # días de mala salud mental (0–30)
    "PhysHlth": 0,             # días de mala salud física (0–30)
    "Age": 2,                  # grupo de edad (ej. 9=60–64)
    "Education": 5,            # nivel educativo (ej. 5=universidad o más)
    "Income": 9,               # nivel de ingresos (ej. 8=40–44k)
    "GenHlth": 4               # autoevaluación salud (1–5)
}])

In [27]:
# Predicción
pred = modelo_pred_diabetes.predict(nuevo_paciente)[0]
proba = modelo_pred_diabetes.predict_proba(nuevo_paciente)[0, 1]

mensaje, accion = presentar_prediccion(proba, thr=0.55, zona_inferior=0.45, zona_superior=0.60, mostrar_score=False)

print("Resultado:", mensaje)
print("Acción sugerida:", accion)

Resultado: NO DIABÉTICO — Riesgo bajo
Acción sugerida: Mantener hábitos saludables y seguimiento rutinario.
